# Work IQ actions

The [Work IQ MCP notebook](workiq-mcp.ipynb) covers read tools. This optional notebook focuses on the four tools that change Microsoft 365 data: `create_entity`, `update_entity`, `delete_entity`, and `do_action`.

**Writes affect real data as the signed-in user.** Each write example is disabled by default and prints its proposed arguments. Review the schema, target, and payload, and obtain human approval before enabling an individual operation. Running all cells unchanged does not create, update, delete, or send anything.

> In the Skillable lab, Work IQ authentication is already configured. For self-deployment, complete the [Work IQ admin setup](../docs/workiq-admin-setup.md) first.

## Step 1: Authenticate as the signed-in user

Load the app registration from `.env`. Sign in with your lab account in the browser when prompted.

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from msal import PublicClientApplication

load_dotenv(override=True)

app_id = os.environ["WORK_IQ_SEARCH_ENTRA_APP_ID"]
tenant_id = os.environ["WORK_IQ_SEARCH_ENTRA_TENANT_ID"]

app = PublicClientApplication(
    client_id=app_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)
auth_result = app.acquire_token_interactive(["api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"])
if "access_token" not in auth_result:
    raise RuntimeError(f"Authentication failed: {auth_result.get('error_description', auth_result)}")

token = auth_result["access_token"]
print("Signed in to Work IQ")

## Step 2: Inspect write tools and schemas

List the write tools and their input schemas, then inspect the create and update schemas for calendar events. These discovery calls do not change data.

- `create_entity` takes a collection's `parentUrl`.
- `update_entity` and `delete_entity` take an item's `entityUrl`.
- `do_action` takes an `actionUrl`, such as `/me/sendMail`.
- `jsonBody` is a JSON-encoded string, so encode payloads with `json.dumps`.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_response = await session.list_tools()
            for tool in tools_response.tools:
                if tool.name in {"create_entity", "update_entity", "delete_entity", "do_action"}:
                    print(tool.name, json.dumps(tool.inputSchema, indent=2))

            for operation in ("create", "update"):
                result = await session.call_tool("get_schema", {"path": "/me/events", "operationType": operation})
                if result.isError:
                    raise RuntimeError(f"Work IQ get_schema failed: {result.content}")
                print(f"Calendar event {operation} schema:")
                for content in result.content:
                    if content.type == "text":
                        print(content.text)

## Step 3: Draft a calendar event

Preview an event for tomorrow with no attendees. After reviewing the schema and payload, set `CREATE = True` only if you approve creating it. Inspect the response and copy the created event's ID for the update and cleanup examples. Do not use an unrelated existing event.

In [ ]:
from datetime import datetime, timedelta, timezone

CREATE = False
start = (datetime.now(timezone.utc) + timedelta(days=1)).replace(hour=15, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)
create_args = {
    "parentUrl": "/me/events",
    "jsonBody": json.dumps({
        "subject": "Work IQ lab practice",
        "start": {"dateTime": start.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
        "end": {"dateTime": end.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
    }),
}
print(json.dumps(create_args, indent=2))

if CREATE:
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool("create_entity", create_args)
    if result.isError:
        raise RuntimeError(f"Work IQ create_entity failed: {result.content}")
    print(result.model_dump_json(indent=2))
else:
    print("Dry run: no event created. Review and approve before setting CREATE = True.")

## Step 4: Update the practice event

If you created the practice event, paste its ID into `event_id`. Review the proposed subject change before setting `UPDATE = True`. Leave the flag off to preview only.

In [ ]:
UPDATE = False
event_id = ""
update_args = {
    "entityUrl": f"/me/events/{event_id}",
    "jsonBody": json.dumps({"subject": "Work IQ lab practice (updated)"}),
}
print(json.dumps(update_args, indent=2))

if UPDATE:
    if not event_id.strip():
        raise ValueError("Set event_id to the practice event you created before enabling UPDATE.")
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool("update_entity", update_args)
    if result.isError:
        raise RuntimeError(f"Work IQ update_entity failed: {result.content}")
    print(result.model_dump_json(indent=2))
else:
    print("Dry run: no event updated. Review and approve before setting UPDATE = True.")

## Step 5: Clean up the practice event

Review `event_id` from the preceding cell and confirm it identifies only the event you created for this exercise. Set `DELETE = True` only after approving its deletion.

In [ ]:
DELETE = False
delete_args = {"entityUrl": f"/me/events/{event_id}"}
print(json.dumps(delete_args, indent=2))

if DELETE:
    if not event_id.strip():
        raise ValueError("Set event_id to the practice event you created before enabling DELETE.")
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool("delete_entity", delete_args)
    if result.isError:
        raise RuntimeError(f"Work IQ delete_entity failed: {result.content}")
    print(result.model_dump_json(indent=2))
else:
    print("Dry run: no event deleted. Review and approve before setting DELETE = True.")

## Step 6: Preview a send-mail action

`do_action` invokes an action such as sending mail, replying, or accepting an event. Inspect its tool schema from step 2 and the [Microsoft Graph sendMail request](https://learn.microsoft.com/graph/api/user-sendmail) before approving the payload.

For this exercise, use your own lab email address as the recipient. Review the address and draft before setting `SEND = True`. Leave it off to avoid sending mail.

In [ ]:
SEND = False
recipient = ""
send_args = {
    "actionUrl": "/me/sendMail",
    "jsonBody": json.dumps({
        "message": {
            "subject": "Work IQ lab practice",
            "body": {"contentType": "Text", "content": "This is a practice message from the Work IQ lab."},
            "toRecipients": [{"emailAddress": {"address": recipient}}],
        },
        "saveToSentItems": True,
    }),
}
print(json.dumps(send_args, indent=2))

if SEND:
    if not recipient.strip():
        raise ValueError("Set recipient to your own lab email address before enabling SEND.")
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool("do_action", send_args)
    if result.isError:
        raise RuntimeError(f"Work IQ do_action failed: {result.content}")
    print(result.model_dump_json(indent=2))
else:
    print("Dry run: no mail sent. Review and approve before setting SEND = True.")

## Recap

The write workflow is **inspect schema -> draft -> confirm -> write**. You previewed the four mutation tools without executing them by default. If you enabled event creation, use the cleanup step to remove only your practice event.

**Optional next step:** [Work IQ A2A](workiq-a2a.ipynb) demonstrates delegating a task to Work IQ as a remote agent.